# 10 — Kategorien-Queries (Teilqueries pro Oberkategorie)

Liest [`data/accounts.csv`](../data/accounts.csv) und schreibt **eine TXT-Datei pro
Oberkategorie** nach [`output/queries/categories/`](../output/queries/categories/).

Format pro TXT: nur `OR`-Strings (keine `language:de`-Hülle), durch
`<<< sub_name[i] — N Handles, M chars >>>`-Marker getrennt — als **Teilqueries**
zum einzelnen Reinkopieren in Brandwatch.

**Logik:**
- Pro Oberkategorie: 1 TXT-Datei.
- Innerhalb der TXT: pro Unterkategorie 1+ OR-Block aus `author:"handle"`-Items.
- **Hard limit: 3 900 Zeichen pro OR-Block.** Wird er länger, wird die Unterkategorie
  in `<sub>1`, `<sub>2`, … aufgeteilt; erst alle Blöcke einer Unterkategorie, dann
  beginnt die nächste.
- **Kanäle:** `{x, instagram, facebook}`, dedup auf `(channel, handle)`.

**Oberkategorien:**
1. **Journalisten** → 1 TXT, alle Journalist:innen aufgelistet.
2. **Politik pro Partei** → 1 TXT pro Bundestagspartei. Filter: `label == <Partei> AND category != "MdB"`.
   Stiftungen bleiben drin (sie sind `category == Organisation` mit `label == <Partei>`).
3. **News (ohne Journalist:innen)** → 1 TXT, Sub-Blöcke pro unique Label
   (Entertainment, Nachrichtenagentur, Nachrichtenprogramm, Online_Only, Rundfunksender, Zeitung).
4. **MdBs** → 1 TXT, Sub-Blöcke pro Partei. Filter: `category == "MdB"`.

In [1]:
import os
import re
from typing import Callable

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
QUERIES_DIR  = os.path.join(PROJECT_ROOT, "output", "queries", "categories")
ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")

ALLOWED_CHANNELS = ["x", "instagram", "facebook"]
MAX_BLOCK_CHARS  = 3900

os.makedirs(QUERIES_DIR, exist_ok=True)

## 1. Daten laden + Helper

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)
print(f"Total accounts: {len(accounts)}")


def bw_author(handle: str) -> str:
    h = str(handle).strip().replace('"', '\\"')
    return f'author:"{h}"'


def chunk_or_string(handles: list[str], max_chars: int = MAX_BLOCK_CHARS) -> list[str]:
    """Greedy-Split: joint Handles als ' OR '-String, neuer Chunk sobald der nächste Handle die Länge > max_chars heben würde."""
    sep = " OR "
    chunks: list[list[str]] = [[]]
    cur_len = 0
    for h in handles:
        item = bw_author(h)
        added = (len(sep) if chunks[-1] else 0) + len(item)
        if chunks[-1] and cur_len + added > max_chars:
            chunks.append([])
            cur_len = 0
            added = len(item)
        chunks[-1].append(item)
        cur_len += added
    return [sep.join(c) for c in chunks if c]


def slugify(s: str) -> str:
    s = s.lower().strip()
    for k, v in {"ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"}.items():
        s = s.replace(k, v)
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


def sort_handles(s: pd.Series) -> list[str]:
    return s.sort_values(key=lambda x: x.str.lower()).tolist()


def by_predicate(predicate: Callable[[pd.DataFrame], pd.Series]) -> Callable[[pd.DataFrame], pd.DataFrame]:
    """Closure-Factory: nimmt ein boolesches Prädikat, liefert eine Filter-Closure die Kanal einschränkt + dedupliziert."""
    def _f(df: pd.DataFrame) -> pd.DataFrame:
        m = predicate(df) & df["channel"].isin(ALLOWED_CHANNELS)
        return (
            df[m]
            .dropna(subset=["handle"])
            .drop_duplicates(subset=["channel", "handle"])
        )
    return _f


def f_journalists() -> Callable[[pd.DataFrame], pd.DataFrame]:
    return by_predicate(lambda d: d["category"].eq("News") & d["label"].eq("Journalist"))


def f_party_no_mdb(party: str) -> Callable[[pd.DataFrame], pd.DataFrame]:
    return by_predicate(lambda d: d["label"].eq(party) & d["category"].ne("MdB"))


def f_news_label(label: str) -> Callable[[pd.DataFrame], pd.DataFrame]:
    return by_predicate(lambda d: d["category"].eq("News") & d["label"].eq(label))


def f_mdb_party(party: str) -> Callable[[pd.DataFrame], pd.DataFrame]:
    return by_predicate(lambda d: d["category"].eq("MdB") & d["label"].eq(party))

Total accounts: 15828


## 2. Konfiguration: Oberkategorien

Eintrag-Format: `(name, filename, [(sub_name, filter_fn), …])`.

- `name` — menschenlesbarer Name (Logging).
- `filename` — Output-TXT in `output/queries/categories/`.
- `sub_name` — Block-Marker-Präfix (`<<< sub_name[i] — … >>>`).
- `filter_fn(df) -> DataFrame` — Filter-Closure auf `accounts` (Kanal/Dedup macht der Helper).

Helper-Closures: `f_journalists()`, `f_party_no_mdb(party)`, `f_news_label(label)`, `f_mdb_party(party)`.

In [3]:
PARTIES = ["AfD", "BSW", "CDU", "CSU", "FDP", "Grüne", "Linke", "SPD", "Sonstige Parteien"]

NEWS_LABELS = [
    "Entertainment",
    "Nachrichtenagentur",
    "Nachrichtenprogramm",
    "Online_Only",
    "Rundfunksender",
    "Zeitung",
]

OBERCATEGORIES: list[tuple[str, str, list[tuple[str, Callable]]]] = [
    # 1) Journalist:innen — eigene TXT, alle aufgelistet
    ("Journalisten", "journalisten_query.txt", [("journalisten", f_journalists())]),

    # 2) Politik pro Partei — eine TXT pro Partei (ohne MdBs, Stiftungen sind drin via label=<Partei>)
    *[
        (
            f"Politik {party}",
            f"politik_{slugify(party)}_query.txt",
            [(slugify(party), f_party_no_mdb(party))],
        )
        for party in PARTIES
    ],

    # 3) News (ohne Journalist:innen) — 1 TXT, Sub-Blöcke pro unique Label
    (
        "News (ohne Journalisten)",
        "news_query.txt",
        [(slugify(lbl), f_news_label(lbl)) for lbl in NEWS_LABELS],
    ),

    # 4) MdBs — 1 TXT, Sub-Blöcke pro Partei
    (
        "MdBs",
        "mdbs_query.txt",
        [(slugify(party), f_mdb_party(party)) for party in PARTIES],
    ),
]

for ober, fname, subs in OBERCATEGORIES:
    sub_names = ", ".join(s[0] for s in subs)
    print(f"{ober:<28s} → {fname:<40s}  subs: {sub_names}")

Journalisten                 → journalisten_query.txt                    subs: journalisten
Politik AfD                  → politik_afd_query.txt                     subs: afd
Politik BSW                  → politik_bsw_query.txt                     subs: bsw
Politik CDU                  → politik_cdu_query.txt                     subs: cdu
Politik CSU                  → politik_csu_query.txt                     subs: csu
Politik FDP                  → politik_fdp_query.txt                     subs: fdp
Politik Grüne                → politik_gruene_query.txt                  subs: gruene
Politik Linke                → politik_linke_query.txt                   subs: linke
Politik SPD                  → politik_spd_query.txt                     subs: spd
Politik Sonstige Parteien    → politik_sonstige_parteien_query.txt       subs: sonstige_parteien
News (ohne Journalisten)     → news_query.txt                            subs: entertainment, nachrichtenagentur, nachrichtenprogramm, online_

## 3. Blöcke bauen + TXT schreiben

In [4]:
def build_sub_blocks(sub_name: str, handles: list[str]) -> list[str]:
    """Eine Unterkategorie → Liste von TXT-Blöcken `<<< sub_name[i] — N Handles, M chars >>>\\n(...)`."""
    if not handles:
        return []
    chunks = chunk_or_string(handles, MAX_BLOCK_CHARS)
    multi = len(chunks) > 1
    out: list[str] = []
    for i, c in enumerate(chunks, 1):
        n = c.count('author:"')
        tag = f"{sub_name}{i}" if multi else sub_name
        out.append(f"<<< {tag} — {n} Handles, {len(c)} chars >>>\n({c})")
    return out


summary: list[dict] = []

for ober_name, filename, subcats in OBERCATEGORIES:
    print(f"\n=== {ober_name} → {filename} ===")
    blocks: list[str] = []
    for sub_name, filter_fn in subcats:
        sub_df = filter_fn(accounts)
        handles = sort_handles(sub_df["handle"])
        if not handles:
            print(f"  {sub_name:<24s}     0 Handles — übersprungen")
            continue
        sub_blocks = build_sub_blocks(sub_name, handles)
        blocks.extend(sub_blocks)
        print(f"  {sub_name:<24s} {len(handles):>5} Handles → {len(sub_blocks):>3} Block(s)")

    out_path = os.path.join(QUERIES_DIR, filename)
    if not blocks:
        print("  ⚠️  Keine Blöcke — Datei wird nicht geschrieben.")
        continue

    text = "\n\n".join(blocks) + "\n"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)

    size = os.path.getsize(out_path)
    n_handles = sum(b.count('author:"') for b in blocks)
    max_block = max(len(b.split("\n", 1)[1]) - 2 for b in blocks)  # OR-String ohne führende '(' und ')'
    summary.append({
        "Oberkategorie": ober_name,
        "Datei": filename,
        "Handles": n_handles,
        "Blöcke": len(blocks),
        "max_block_chars": max_block,
        "Bytes": size,
    })
    print(f"  → {size:,} bytes, {n_handles} Handles, {len(blocks)} Block(s), max OR-Block: {max_block} chars")


=== Journalisten → journalisten_query.txt ===
  journalisten              1832 Handles →  12 Block(s)
  → 44,967 bytes, 1832 Handles, 12 Block(s), max OR-Block: 3895 chars

=== Politik AfD → politik_afd_query.txt ===
  afd                        693 Handles →   5 Block(s)
  → 19,401 bytes, 693 Handles, 5 Block(s), max OR-Block: 3889 chars

=== Politik BSW → politik_bsw_query.txt ===
  bsw                         99 Handles →   1 Block(s)
  → 2,719 bytes, 99 Handles, 1 Block(s), max OR-Block: 2677 chars

=== Politik CDU → politik_cdu_query.txt ===
  cdu                       1451 Handles →  11 Block(s)
  → 39,424 bytes, 1451 Handles, 11 Block(s), max OR-Block: 3896 chars

=== Politik CSU → politik_csu_query.txt ===
  csu                        224 Handles →   2 Block(s)
  → 6,210 bytes, 224 Handles, 2 Block(s), max OR-Block: 3872 chars

=== Politik FDP → politik_fdp_query.txt ===
  fdp                        565 Handles →   4 Block(s)
  → 14,535 bytes, 565 Handles, 4 Block(s), max OR-B

## 4. Übersicht

In [5]:
if summary:
    s = pd.DataFrame(summary)
    print(s.to_string(index=False))
    over = s[s["max_block_chars"] > MAX_BLOCK_CHARS]
    if not over.empty:
        print(f"\n⚠️  {len(over)} Oberkategorie(n) mit Blöcken > {MAX_BLOCK_CHARS} chars (Einzel-Handle länger als Limit?).")
    else:
        print(f"\n✓ Alle OR-Blöcke ≤ {MAX_BLOCK_CHARS} chars.")
    print(f"Gesamt: {s['Handles'].sum()} Handles, {s['Blöcke'].sum()} Blöcke, {s['Bytes'].sum():,} bytes über {len(s)} Datei(en).")
else:
    print("Keine Outputs erzeugt.")

            Oberkategorie                               Datei  Handles  Blöcke  max_block_chars  Bytes
             Journalisten              journalisten_query.txt     1832      12             3895  44967
              Politik AfD               politik_afd_query.txt      693       5             3889  19401
              Politik BSW               politik_bsw_query.txt       99       1             2677   2719
              Politik CDU               politik_cdu_query.txt     1451      11             3896  39424
              Politik CSU               politik_csu_query.txt      224       2             3872   6210
              Politik FDP               politik_fdp_query.txt      565       4             3899  14535
            Politik Grüne            politik_gruene_query.txt     1083       8             3896  28895
            Politik Linke             politik_linke_query.txt      367       3             3900  10210
              Politik SPD               politik_spd_query.txt     1542   